# 📊 Praktikum Machine Learning
## Regresi Linear, Regresi Logistik & Klasifikasi KNN

**Nama:** [Isi Nama Anda]

**NIM:** [Isi NIM Anda]

---

### Daftar Isi:
1. Regresi Linear Sederhana
2. Regresi Linear Berganda
3. Regresi Polinomial
4. Regresi Logistik
5. Klasifikasi KNN (data buatan)
6. Klasifikasi KNN (data Kaggle)

---
## 📦 Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, confusion_matrix,
    classification_report
)
from sklearn.neighbors import KNeighborsClassifier

import warnings
warnings.filterwarnings('ignore')

print('✅ Semua library berhasil diimport!')

---
## 1️⃣ REGRESI LINEAR SEDERHANA

**Definisi:** Regresi linear sederhana adalah metode statistik untuk memodelkan hubungan **linear antara satu variabel independen (X) dan satu variabel dependen (Y)**.

**Rumus:** `Y = b0 + b1*X + ε`

**Contoh Kasus:** Memprediksi **nilai ujian mahasiswa** berdasarkan **jam belajar per hari**.

In [ ]:
# ── DATA BUATAN: Jam Belajar vs Nilai Ujian ──
np.random.seed(42)
jam_belajar = np.array([1, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 7, 8, 9, 10])
nilai_ujian = np.array([40, 50, 53, 58, 60, 65, 67, 72, 74, 78, 83, 88, 92, 96])

# Buat DataFrame
df_simple = pd.DataFrame({'Jam_Belajar': jam_belajar, 'Nilai_Ujian': nilai_ujian})
print(df_simple.to_string(index=False))

In [ ]:
# ── TRAINING MODEL ──
X_s = df_simple[['Jam_Belajar']]
y_s = df_simple['Nilai_Ujian']

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_s, y_s, test_size=0.3, random_state=42
)

model_simple = LinearRegression()
model_simple.fit(X_train_s, y_train_s)

y_pred_s = model_simple.predict(X_test_s)

print(f'📌 Intercept (b0)  : {model_simple.intercept_:.4f}')
print(f'📌 Koefisien (b1)  : {model_simple.coef_[0]:.4f}')
print(f'📌 Persamaan       : Nilai = {model_simple.intercept_:.2f} + {model_simple.coef_[0]:.2f} * Jam_Belajar')
print(f'\n📊 MSE  : {mean_squared_error(y_test_s, y_pred_s):.4f}')
print(f'📊 RMSE : {np.sqrt(mean_squared_error(y_test_s, y_pred_s)):.4f}')
print(f'📊 R²   : {r2_score(y_test_s, y_pred_s):.4f}')

In [ ]:
# ── VISUALISASI ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: Scatter + Garis Regresi
x_line = np.linspace(jam_belajar.min(), jam_belajar.max(), 100).reshape(-1, 1)
y_line = model_simple.predict(x_line)

axes[0].scatter(X_train_s, y_train_s, color='steelblue', label='Data Train', zorder=5)
axes[0].scatter(X_test_s, y_test_s, color='tomato', label='Data Test', zorder=5)
axes[0].plot(x_line, y_line, color='green', linewidth=2, label='Garis Regresi')
axes[0].set_title('Regresi Linear Sederhana\nJam Belajar vs Nilai Ujian', fontsize=12)
axes[0].set_xlabel('Jam Belajar per Hari')
axes[0].set_ylabel('Nilai Ujian')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Actual vs Predicted
axes[1].scatter(y_test_s, y_pred_s, color='purple', zorder=5)
axes[1].plot([y_test_s.min(), y_test_s.max()], [y_test_s.min(), y_test_s.max()],
             'r--', label='Ideal (y=x)')
axes[1].set_title('Nilai Aktual vs Prediksi', fontsize=12)
axes[1].set_xlabel('Nilai Aktual')
axes[1].set_ylabel('Nilai Prediksi')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 📝 Analisis Regresi Linear Sederhana

- **Persamaan regresi** yang dihasilkan menunjukkan bahwa setiap **penambahan 1 jam belajar**, nilai ujian meningkat sekitar **nilai koefisien b1**.
- **R² mendekati 1** berarti variabel jam belajar mampu menjelaskan sebagian besar variasi nilai ujian — hubungan keduanya sangat kuat.
- Dari scatter plot *Actual vs Predicted*, titik-titik mendekati garis merah putus-putus (y=x) menandakan model cukup akurat.
- **Kesimpulan:** Semakin banyak jam belajar, semakin tinggi nilai ujian — sesuai ekspektasi secara logis.

---
## 2️⃣ REGRESI LINEAR BERGANDA

**Definisi:** Regresi linear berganda memodelkan hubungan antara **dua atau lebih variabel independen** dengan satu variabel dependen.

**Rumus:** `Y = b0 + b1*X1 + b2*X2 + ... + bn*Xn + ε`

**Contoh Kasus:** Memprediksi **harga rumah** berdasarkan luas tanah, jumlah kamar, dan jarak ke pusat kota.

In [ ]:
# ── DATA BUATAN: Harga Rumah ──
np.random.seed(7)
n = 80

luas_tanah   = np.random.randint(60, 300, n)          # m²
jumlah_kamar = np.random.randint(1, 6, n)              # kamar
jarak_kota   = np.random.uniform(1, 30, n)             # km

# Harga dipengaruhi ketiga faktor + noise
harga_rumah  = (luas_tanah * 3.5 +
                jumlah_kamar * 80 -
                jarak_kota * 10 +
                np.random.normal(0, 50, n))             # juta rupiah

df_multi = pd.DataFrame({
    'Luas_Tanah_m2': luas_tanah,
    'Jumlah_Kamar': jumlah_kamar,
    'Jarak_Kota_km': jarak_kota.round(1),
    'Harga_Juta': harga_rumah.round(1)
})

print(f'Shape data: {df_multi.shape}')
print(df_multi.head(10).to_string(index=False))
print('\nStatistik Deskriptif:')
print(df_multi.describe().round(2))

In [ ]:
# ── TRAINING MODEL ──
X_m = df_multi[['Luas_Tanah_m2', 'Jumlah_Kamar', 'Jarak_Kota_km']]
y_m = df_multi['Harga_Juta']

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_m, y_m, test_size=0.2, random_state=42
)

model_multi = LinearRegression()
model_multi.fit(X_train_m, y_train_m)

y_pred_m = model_multi.predict(X_test_m)

print(f'📌 Intercept       : {model_multi.intercept_:.4f}')
for nama, koef in zip(X_m.columns, model_multi.coef_):
    print(f'📌 Koef {nama:20s}: {koef:.4f}')

print(f'\n📊 MSE  : {mean_squared_error(y_test_m, y_pred_m):.4f}')
print(f'📊 RMSE : {np.sqrt(mean_squared_error(y_test_m, y_pred_m)):.4f}')
print(f'📊 R²   : {r2_score(y_test_m, y_pred_m):.4f}')

In [ ]:
# ── VISUALISASI ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

fitur = ['Luas_Tanah_m2', 'Jumlah_Kamar', 'Jarak_Kota_km']
warna = ['steelblue', 'orange', 'green']

for i, (f, w) in enumerate(zip(fitur, warna)):
    axes[i].scatter(df_multi[f], df_multi['Harga_Juta'], alpha=0.5, color=w)
    axes[i].set_xlabel(f)
    axes[i].set_ylabel('Harga (Juta Rp)')
    axes[i].set_title(f'{f}\nvs Harga Rumah')
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Korelasi Setiap Fitur dengan Harga Rumah', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Correlation heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(df_multi.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Heatmap Korelasi Antar Variabel')
plt.tight_layout()
plt.show()

### 📝 Analisis Regresi Linear Berganda

- **Luas Tanah** memiliki koefisien positif terbesar → paling dominan mempengaruhi harga rumah.
- **Jumlah Kamar** juga berpengaruh positif — semakin banyak kamar, semakin mahal.
- **Jarak ke Kota** berkoefisien negatif — semakin jauh dari pusat kota, harga cenderung turun.
- **R² yang tinggi** menunjukkan model mampu menjelaskan variasi harga dengan baik menggunakan 3 variabel ini.
- Dari heatmap korelasi, luas tanah memiliki korelasi paling kuat dengan harga.

---
## 3️⃣ REGRESI POLINOMIAL

**Definisi:** Regresi polinomial adalah bentuk regresi yang menangkap **hubungan non-linear** dengan menambahkan pangkat dari variabel independen.

**Rumus:** `Y = b0 + b1*X + b2*X² + b3*X³ + ε`

**Contoh Kasus:** Memprediksi **konsumsi bahan bakar** berdasarkan kecepatan kendaraan (hubungan melengkung).

In [ ]:
# ── DATA BUATAN: Kecepatan vs Konsumsi BBM ──
# Mobil paling irit di kecepatan menengah (~80 km/h)
np.random.seed(10)
kecepatan = np.linspace(20, 160, 50)
konsumsi  = 0.0008 * (kecepatan - 80)**2 + 7 + np.random.normal(0, 0.5, 50)
# konsumsi L/100km — berbentuk parabola (paling irit di ~80 km/h)

df_poly = pd.DataFrame({'Kecepatan_kmh': kecepatan.round(1), 'Konsumsi_L100km': konsumsi.round(2)})
print(df_poly.head(10).to_string(index=False))

In [ ]:
# ── TRAINING MODEL ──
X_p = df_poly[['Kecepatan_kmh']]
y_p = df_poly['Konsumsi_L100km']

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_p, y_p, test_size=0.2, random_state=42
)

# Linear
lin_model = LinearRegression()
lin_model.fit(X_train_p, y_train_p)
y_pred_lin = lin_model.predict(X_test_p)

# Polinomial derajat 2
poly2 = PolynomialFeatures(degree=2)
X_train_p2 = poly2.fit_transform(X_train_p)
X_test_p2  = poly2.transform(X_test_p)
poly2_model = LinearRegression()
poly2_model.fit(X_train_p2, y_train_p)
y_pred_p2 = poly2_model.predict(X_test_p2)

# Polinomial derajat 3
poly3 = PolynomialFeatures(degree=3)
X_train_p3 = poly3.fit_transform(X_train_p)
X_test_p3  = poly3.transform(X_test_p)
poly3_model = LinearRegression()
poly3_model.fit(X_train_p3, y_train_p)
y_pred_p3 = poly3_model.predict(X_test_p3)

print('Perbandingan R² pada data test:')
print(f'  Linear (derajat 1) : R² = {r2_score(y_test_p, y_pred_lin):.4f}')
print(f'  Poly derajat 2     : R² = {r2_score(y_test_p, y_pred_p2):.4f}')
print(f'  Poly derajat 3     : R² = {r2_score(y_test_p, y_pred_p3):.4f}')

In [ ]:
# ── VISUALISASI ──
x_vis = np.linspace(20, 160, 200).reshape(-1, 1)

y_vis_lin  = lin_model.predict(x_vis)
y_vis_p2   = poly2_model.predict(poly2.transform(x_vis))
y_vis_p3   = poly3_model.predict(poly3.transform(x_vis))

plt.figure(figsize=(10, 5))
plt.scatter(kecepatan, konsumsi, color='gray', alpha=0.6, label='Data Asli', zorder=5)
plt.plot(x_vis, y_vis_lin, 'r-',  label=f'Linear  R²={r2_score(y_test_p, y_pred_lin):.3f}')
plt.plot(x_vis, y_vis_p2,  'b--', label=f'Poly d=2 R²={r2_score(y_test_p, y_pred_p2):.3f}', linewidth=2)
plt.plot(x_vis, y_vis_p3,  'g:',  label=f'Poly d=3 R²={r2_score(y_test_p, y_pred_p3):.3f}', linewidth=2)
plt.xlabel('Kecepatan (km/h)')
plt.ylabel('Konsumsi BBM (L/100km)')
plt.title('Regresi Polinomial: Kecepatan vs Konsumsi BBM')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 📝 Analisis Regresi Polinomial

- Data konsumsi BBM membentuk **kurva parabola** — paling irit pada kecepatan menengah.
- Regresi **linear (garis lurus) gagal** menangkap pola ini, terlihat dari R² yang sangat rendah.
- **Regresi polinomial derajat 2** berhasil mengikuti pola melengkung dengan R² jauh lebih baik.
- **Derajat 3** mungkin sedikit lebih fit, namun risiko *overfitting* meningkat.
- **Kesimpulan:** Gunakan regresi polinomial ketika hubungan antar variabel tidak linear secara visual.

---
## 4️⃣ REGRESI LOGISTIK

**Definisi:** Regresi logistik digunakan untuk **klasifikasi** (output berupa kategori/kelas), bukan regresi numerik. Output berupa **probabilitas** yang dipetakan ke kelas menggunakan fungsi sigmoid.

**Rumus Sigmoid:** `P(Y=1) = 1 / (1 + e^(-z))` di mana `z = b0 + b1*X1 + ...`

**Contoh Kasus:** Memprediksi apakah seorang mahasiswa **lulus atau tidak lulus** berdasarkan nilai tugas dan kehadiran.

In [ ]:
# ── DATA BUATAN: Prediksi Kelulusan Mahasiswa ──
np.random.seed(99)
n = 120

nilai_tugas = np.random.randint(40, 100, n)
persen_hadir = np.random.randint(30, 100, n)

# Aturan lulus: nilai_tugas >= 60 AND hadir >= 70, dengan sedikit noise
prob_lulus = 1 / (1 + np.exp(-(
    0.08 * (nilai_tugas - 60) + 0.05 * (persen_hadir - 70)
)))
lulus = (np.random.rand(n) < prob_lulus).astype(int)

df_logit = pd.DataFrame({
    'Nilai_Tugas': nilai_tugas,
    'Persen_Hadir': persen_hadir,
    'Lulus': lulus
})

print(f'Shape: {df_logit.shape}')
print(f"\nDistribusi kelas:")
print(df_logit['Lulus'].value_counts().rename({0:'Tidak Lulus', 1:'Lulus'}))
print(df_logit.head(10).to_string(index=False))

In [ ]:
# ── TRAINING MODEL ──
X_l = df_logit[['Nilai_Tugas', 'Persen_Hadir']]
y_l = df_logit['Lulus']

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_l, y_l, test_size=0.25, random_state=42
)

scaler_l = StandardScaler()
X_train_l_sc = scaler_l.fit_transform(X_train_l)
X_test_l_sc  = scaler_l.transform(X_test_l)

model_logit = LogisticRegression(random_state=42)
model_logit.fit(X_train_l_sc, y_train_l)

y_pred_l = model_logit.predict(X_test_l_sc)

print(f'📊 Akurasi : {accuracy_score(y_test_l, y_pred_l):.4f} '
      f'({accuracy_score(y_test_l, y_pred_l)*100:.1f}%)')
print()
print('📋 Classification Report:')
print(classification_report(y_test_l, y_pred_l, target_names=['Tidak Lulus', 'Lulus']))

In [ ]:
# ── VISUALISASI ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test_l, y_pred_l)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Tidak Lulus', 'Lulus'],
            yticklabels=['Tidak Lulus', 'Lulus'], ax=axes[0])
axes[0].set_xlabel('Prediksi')
axes[0].set_ylabel('Aktual')
axes[0].set_title('Confusion Matrix\nRegresi Logistik')

# Decision Boundary
xx, yy = np.meshgrid(
    np.linspace(X_l['Nilai_Tugas'].min()-5, X_l['Nilai_Tugas'].max()+5, 200),
    np.linspace(X_l['Persen_Hadir'].min()-5, X_l['Persen_Hadir'].max()+5, 200)
)
grid = scaler_l.transform(np.c_[xx.ravel(), yy.ravel()])
Z = model_logit.predict(grid).reshape(xx.shape)

axes[1].contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
scatter = axes[1].scatter(X_l['Nilai_Tugas'], X_l['Persen_Hadir'],
                           c=y_l, cmap='RdBu', edgecolors='k', s=50)
axes[1].set_xlabel('Nilai Tugas')
axes[1].set_ylabel('Persen Kehadiran')
axes[1].set_title('Decision Boundary\nRegresi Logistik')
plt.colorbar(scatter, ax=axes[1], ticks=[0, 1], label='0=Tidak Lulus, 1=Lulus')

plt.tight_layout()
plt.show()

### 📝 Analisis Regresi Logistik

- Model regresi logistik berhasil mengklasifikasikan mahasiswa lulus/tidak dengan akurasi yang baik.
- Dari **decision boundary**, terlihat ada **garis pemisah** antara mahasiswa yang lulus (biru) dan tidak lulus (merah).
- **Precision** dan **Recall** keduanya perlu diperhatikan — dalam konteks pendidikan, False Negative (prediksi lulus padahal tidak) lebih merugikan.
- **Confusion Matrix** menunjukkan seberapa banyak prediksi yang benar di masing-masing kelas.
- **Perbedaan dengan regresi linear:** Regresi logistik menghasilkan output berupa **probabilitas (0–1)** dan digunakan untuk **klasifikasi**, bukan prediksi nilai kontinu.

---
## 5️⃣ KLASIFIKASI KNN (K-Nearest Neighbors) — Data Buatan

**Definisi:** KNN mengklasifikasikan data baru berdasarkan **K tetangga terdekat** dari data training. Tidak ada fase "training" eksplisit — KNN menyimpan seluruh data (lazy learner).

**Parameter penting:** `K` (jumlah tetangga) — K kecil = overfitting, K besar = underfitting.

**Contoh Kasus:** Klasifikasi jenis bunga berdasarkan panjang dan lebar kelopak (data buatan 3 kelas).

In [ ]:
# ── DATA BUATAN: Klasifikasi 3 Jenis Bunga ──
from sklearn.datasets import make_blobs

X_knn, y_knn = make_blobs(
    n_samples=180,
    centers=[[2,2],[5,8],[9,4]],
    cluster_std=1.2,
    random_state=42
)

nama_kelas = {0: 'Mawar', 1: 'Melati', 2: 'Anggrek'}
df_knn = pd.DataFrame(X_knn, columns=['Panjang_Kelopak', 'Lebar_Kelopak'])
df_knn['Jenis_Bunga'] = [nama_kelas[k] for k in y_knn]

print(df_knn['Jenis_Bunga'].value_counts())
print(df_knn.head(8).to_string(index=False))

In [ ]:
# ── MENCARI K OPTIMAL ──
X_train_k, X_test_k, y_train_k, y_test_k = train_test_split(
    X_knn, y_knn, test_size=0.25, random_state=42
)

scaler_k = StandardScaler()
X_train_k_sc = scaler_k.fit_transform(X_train_k)
X_test_k_sc  = scaler_k.transform(X_test_k)

k_values = range(1, 21)
train_acc = []
test_acc  = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_k_sc, y_train_k)
    train_acc.append(accuracy_score(y_train_k, knn.predict(X_train_k_sc)))
    test_acc.append(accuracy_score(y_test_k, knn.predict(X_test_k_sc)))

k_terbaik = k_values[np.argmax(test_acc)]
print(f'K terbaik: {k_terbaik} dengan Test Accuracy = {max(test_acc):.4f}')

plt.figure(figsize=(9, 4))
plt.plot(k_values, train_acc, 'b-o', label='Train Accuracy', markersize=5)
plt.plot(k_values, test_acc,  'r-s', label='Test Accuracy',  markersize=5)
plt.axvline(x=k_terbaik, color='green', linestyle='--', label=f'K optimal = {k_terbaik}')
plt.xlabel('Nilai K')
plt.ylabel('Akurasi')
plt.title('Mencari K Optimal pada KNN')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── MODEL FINAL DENGAN K TERBAIK ──
knn_final = KNeighborsClassifier(n_neighbors=k_terbaik)
knn_final.fit(X_train_k_sc, y_train_k)
y_pred_knn = knn_final.predict(X_test_k_sc)

print(f'📊 Akurasi KNN (K={k_terbaik}): {accuracy_score(y_test_k, y_pred_knn)*100:.1f}%')
print()
label_names = [nama_kelas[i] for i in range(3)]
print(classification_report(y_test_k, y_pred_knn, target_names=label_names))

In [ ]:
# ── VISUALISASI DECISION BOUNDARY KNN ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter data
warna_kelas = ['red', 'blue', 'green']
for cls in range(3):
    mask = y_knn == cls
    axes[0].scatter(X_knn[mask, 0], X_knn[mask, 1],
                    c=warna_kelas[cls], label=nama_kelas[cls], edgecolors='k', alpha=0.7)
axes[0].set_title('Distribusi Data Bunga')
axes[0].set_xlabel('Panjang Kelopak')
axes[0].set_ylabel('Lebar Kelopak')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Decision boundary
xx2, yy2 = np.meshgrid(
    np.linspace(X_knn[:,0].min()-1, X_knn[:,0].max()+1, 200),
    np.linspace(X_knn[:,1].min()-1, X_knn[:,1].max()+1, 200)
)
grid2 = scaler_k.transform(np.c_[xx2.ravel(), yy2.ravel()])
Z2 = knn_final.predict(grid2).reshape(xx2.shape)

axes[1].contourf(xx2, yy2, Z2, alpha=0.25, cmap='RdYlGn')
for cls in range(3):
    mask = y_knn == cls
    axes[1].scatter(X_knn[mask, 0], X_knn[mask, 1],
                    c=warna_kelas[cls], label=nama_kelas[cls], edgecolors='k', alpha=0.7)
axes[1].set_title(f'Decision Boundary KNN (K={k_terbaik})')
axes[1].set_xlabel('Panjang Kelopak')
axes[1].set_ylabel('Lebar Kelopak')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Klasifikasi KNN — Jenis Bunga (Data Buatan)', fontsize=13)
plt.tight_layout()
plt.show()

### 📝 Analisis KNN (Data Buatan)

- KNN berhasil memisahkan ketiga jenis bunga dengan akurasi tinggi karena datanya terkluster dengan baik.
- Dari grafik **K vs Akurasi**, terbukti bahwa K terlalu kecil (overfitting) atau terlalu besar (underfitting) mengurangi performa.
- **Decision boundary** KNN tidak berupa garis lurus, melainkan fleksibel mengikuti bentuk data.
- Menggunakan **StandardScaler penting** — tanpa scaling, fitur dengan skala besar akan mendominasi perhitungan jarak.

---
## 6️⃣ KLASIFIKASI KNN — Dataset Kaggle (Iris / Diabetes)

**Dataset:** Dataset Iris dari Kaggle / sklearn (klasifikasi jenis bunga Iris)

> Dataset Iris adalah benchmark klasik dalam machine learning, tersedia di Kaggle:
> https://www.kaggle.com/datasets/uciml/iris
> 
> Disini kita load langsung dari sklearn untuk kemudahan (data identik).

In [ ]:
# ── LOAD DATASET IRIS (KAGGLE) ──
from sklearn.datasets import load_iris

iris = load_iris()
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['species'] = [iris.target_names[t] for t in iris.target]
df_iris['target']  = iris.target

print('Shape:', df_iris.shape)
print('\nKelas:')
print(df_iris['species'].value_counts())
print('\nSample data:')
print(df_iris.head(8).to_string(index=False))
print('\nStatistik Deskriptif:')
print(df_iris.describe().round(2))

In [ ]:
# ── EKSPLORASI DATA ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pairplot manual (2 fitur paling informatif)
warna_iris = {'setosa': 'red', 'versicolor': 'blue', 'virginica': 'green'}
for sp, col in warna_iris.items():
    subset = df_iris[df_iris['species'] == sp]
    axes[0].scatter(subset['petal length (cm)'], subset['petal width (cm)'],
                    c=col, label=sp, edgecolors='k', alpha=0.7)
axes[0].set_xlabel('Petal Length (cm)')
axes[0].set_ylabel('Petal Width (cm)')
axes[0].set_title('Iris: Petal Length vs Petal Width')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Boxplot
df_melt = df_iris.melt(id_vars='species',
                        value_vars=iris.feature_names,
                        var_name='Fitur', value_name='Nilai')
sns.boxplot(data=df_melt, x='Fitur', y='Nilai', hue='species', ax=axes[1])
axes[1].set_title('Distribusi Fitur per Kelas')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# ── TRAINING KNN ──
X_iris = iris.data
y_iris = iris.target

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

scaler_i = StandardScaler()
X_train_i_sc = scaler_i.fit_transform(X_train_i)
X_test_i_sc  = scaler_i.transform(X_test_i)

# Cari K optimal
k_vals = range(1, 21)
test_accs = []
for k in k_vals:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train_i_sc, y_train_i)
    test_accs.append(accuracy_score(y_test_i, m.predict(X_test_i_sc)))

k_opt = k_vals[np.argmax(test_accs)]
print(f'K optimal: {k_opt}, Test Accuracy: {max(test_accs)*100:.1f}%')

# Model final
knn_iris = KNeighborsClassifier(n_neighbors=k_opt)
knn_iris.fit(X_train_i_sc, y_train_i)
y_pred_iris = knn_iris.predict(X_test_i_sc)

print()
print('📋 Classification Report:')
print(classification_report(y_test_i, y_pred_iris, target_names=iris.target_names))

In [ ]:
# ── VISUALISASI HASIL ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm_iris = confusion_matrix(y_test_i, y_pred_iris)
sns.heatmap(cm_iris, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names, ax=axes[0])
axes[0].set_xlabel('Prediksi')
axes[0].set_ylabel('Aktual')
axes[0].set_title('Confusion Matrix\nKNN Dataset Iris')

# Accuracy per K
axes[1].plot(k_vals, test_accs, 'b-o', markersize=5)
axes[1].axvline(x=k_opt, color='red', linestyle='--', label=f'K optimal = {k_opt}')
axes[1].set_xlabel('Nilai K')
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('K vs Akurasi (Dataset Iris)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\n🎯 Akurasi Final KNN Iris (K={k_opt}): {accuracy_score(y_test_i, y_pred_iris)*100:.1f}%')

### 📝 Analisis KNN — Dataset Iris (Kaggle)

- **Dataset Iris** memiliki 150 sampel dengan 3 kelas (Setosa, Versicolor, Virginica) dan 4 fitur.
- Dari boxplot, fitur **petal length** dan **petal width** paling informatif untuk membedakan kelas.
- KNN mencapai akurasi sangat tinggi (~95–97%) karena ketiga kelas cukup terpisah.
- Kelas **Setosa** paling mudah diklasifikasikan — tidak ada misklasifikasi di confusion matrix.
- **Versicolor** dan **Virginica** kadang tertukar karena overlap fitur yang lebih besar.
- **Kesimpulan:** KNN adalah algoritma sederhana namun efektif, terutama ketika fitur sudah di-scale dan data tidak terlalu besar.

---
## 📌 RINGKASAN & PERBANDINGAN MODEL

| Model | Tipe | Variabel Output | Contoh Kasus |
|---|---|---|---|
| Regresi Linear Sederhana | Prediksi | Numerik kontinu | Jam belajar → Nilai ujian |
| Regresi Linear Berganda | Prediksi | Numerik kontinu | Fitur rumah → Harga |
| Regresi Polinomial | Prediksi | Numerik kontinu | Kecepatan → Konsumsi BBM |
| Regresi Logistik | Klasifikasi | Kategori (0/1) | Nilai & kehadiran → Lulus? |
| KNN | Klasifikasi | Kategori (multi) | Fitur bunga → Jenis bunga |

### 💡 Kesimpulan Umum:

1. **Regresi linear** cocok untuk hubungan **linier dan output numerik**.
2. **Regresi polinomial** digunakan ketika pola data **melengkung/non-linear**.
3. **Regresi logistik** meskipun namanya 'regresi', sebenarnya untuk **klasifikasi biner**.
4. **KNN** bersifat **non-parametrik dan lazy learner** — performa sangat bergantung pada K dan skala data.
5. **Normalisasi/Standardisasi** fitur sangat penting untuk model berbasis jarak (KNN).
6. Selalu evaluasi model dengan **R² (regresi)** dan **Accuracy/F1 (klasifikasi)** + visualisasi.